In [ ]:
"""
Notebook to evaluate effect of StratifiedKFold vs StratifiedGroupKFold in cross-validation for EpiAtlas dataset.
"""
# pylint: disable=redefined-outer-name, import-error
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict

import numpy as np
from imblearn.over_sampling import RandomOverSampler
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold

from epiclass.core import metadata
from epiclass.core.data_source import EpiDataSource
from epiclass.core.lazy.lazy_fold_factory import (
    LazyEpiAtlasFoldFactory as EpiAtlasFoldFactory,
    LazyEpiAtlasMetadata as EpiAtlasMetadata,
)
from epiclass.utils import modify_metadata

In [ ]:
ASSAY = "assay_epiclass"
CELL_TYPE = "harmonized_sample_ontology_intermediate"
EPIRR = "epirr_id"
UUID = "uuid"

In [ ]:
metadata_path = (
    Path.home()
    / "Projects/epiclass/output/paper/data/metadata/epiatlas/hg38_2023-epiatlas-dfreeze-pospurge-nodup_filterCtl.json"
)
md5_filepath = metadata_path.parent / f"{metadata_path.stem}.md5"
chromsize_path = (
    Path.home() / "Projects/epiclass/output/paper/data/chromsizes/hg38.noy.chrom.sizes"
)

In [ ]:
datasource = EpiDataSource(
    metadata=metadata_path,
    chromsize=chromsize_path,
    hdf5=md5_filepath,
)

In [ ]:
category = CELL_TYPE
min_class_size = 10

In [ ]:
my_metadata = metadata.UUIDMetadata(datasource.metadata_file)

# --- Prefilter metadata ---
my_metadata.remove_category_subsets(label_category="track_type", labels=["Unique.raw"])
my_metadata.remove_missing_labels(category)

label_list = metadata.env_filtering(my_metadata, category)

if any(
    label in category
    for label in set(
        [
            "harmonized_sample_ontology_intermediate",
            "harm_sample_ontology_intermediate",
            "cell_type",
        ]
    )
):
    categories = set(my_metadata.get_categories())
    if "assay_epiclass" in categories:
        assay_cat = "assay_epiclass"
    elif "assay" in categories:
        assay_cat = "assay"
    else:
        raise ValueError("Cannot find assay category for class pairs.")
    my_metadata = modify_metadata.filter_by_pairs(
        my_metadata=my_metadata,
        assay_cat=assay_cat,
        cat2=category,
        nb_pairs=9,
        min_per_pair=10,
    )

In [ ]:
def epirr_per_class(
    meta: metadata.UUIDMetadata, label_category: str
) -> Dict[str, set[str]]:
    """Return {label/class:uuid list} dict for a given metadata category.

    Can fail if remove_missing_labels has not been ran before.
    """
    epirr_dict = defaultdict(set)
    for md5 in meta.signal_ids:
        label = meta[md5][label_category]
        epirr = meta[md5][EPIRR]

        epirr_dict[label].add(epirr)
    return epirr_dict

In [ ]:
my_metadata.display_labels(category)

In [ ]:
full_dataset = EpiAtlasMetadata(
    datasource=datasource,
    metadata=my_metadata,
    label_category=category,
    label_list=label_list,
    min_class_size=10,
    force_filter=True,
)
dataset_handler = EpiAtlasFoldFactory(
    epiatlas_dataset=full_dataset,
    n_fold=10,
    test_ratio=0,
)

In [ ]:
epirrs_dist = epirr_per_class(full_dataset.metadata, category)

In [ ]:
epirr_count = Counter()
for label, epirr_list in epirrs_dist.items():
    epirr_count[label] = len(epirr_list)

In [ ]:
print("Number of unique epirr_id per class:")
for label, count in epirr_count.most_common():
    print(f"{label}: {count}")
print(f"Total unique epirr_id: {sum(epirr_count.values())}")

In [ ]:
classes = sorted(my_metadata.uuid_counter(category).keys())
display(classes)

With `scikit-learn==1.5.2`, three validation folds has a missing class. I think this was solved in a later version (Oct 28, 2025): https://github.com/scikit-learn/scikit-learn/pull/32540. It now works in `1.8.0`.

<details>
<summary>Previous results</summary>

```text
--- Fold 0 ---
Train: 14919 samples, 5339 UUIDs, 1694 EpiRRs
Valid: 1460 samples, 527 UUIDs, 188 EpiRRs
--- Fold 1 ---
Train: 14670 samples, 5255 UUIDs, 1694 EpiRRs
Valid: 1709 samples, 611 UUIDs, 188 EpiRRs
--- Fold 2 ---
Train: 14691 samples, 5260 UUIDs, 1692 EpiRRs
Valid: 1688 samples, 606 UUIDs, 190 EpiRRs
--- Fold 3 ---
Train: 14602 samples, 5236 UUIDs, 1692 EpiRRs
Valid: 1777 samples, 630 UUIDs, 190 EpiRRs
WARNING: Class 'mammary gland epithelial cell' has no validation samples!
Class 'mammary gland epithelial cell': Train 387 samples, Valid 0 samples
--- Fold 4 ---
Train: 14814 samples, 5303 UUIDs, 1693 EpiRRs
Valid: 1565 samples, 563 UUIDs, 189 EpiRRs
WARNING: Class 'endoderm-derived structure' has no validation samples!
Class 'endoderm-derived structure': Train 584 samples, Valid 0 samples
--- Fold 5 ---
Train: 14839 samples, 5311 UUIDs, 1694 EpiRRs
Valid: 1540 samples, 555 UUIDs, 188 EpiRRs
--- Fold 6 ---
Train: 14600 samples, 5234 UUIDs, 1693 EpiRRs
Valid: 1779 samples, 632 UUIDs, 189 EpiRRs
--- Fold 7 ---
Train: 14818 samples, 5304 UUIDs, 1695 EpiRRs
Valid: 1561 samples, 562 UUIDs, 187 EpiRRs
--- Fold 8 ---
Train: 14778 samples, 5293 UUIDs, 1696 EpiRRs
Valid: 1601 samples, 573 UUIDs, 186 EpiRRs
--- Fold 9 ---
Train: 14680 samples, 5259 UUIDs, 1695 EpiRRs
Valid: 1699 samples, 607 UUIDs, 187 EpiRRs
WARNING: Class 'hepatocyte' has no validation samples!
Class 'hepatocyte': Train 434 samples, Valid 0 samples
```

</details>

In [ ]:
for i, dset in enumerate(dataset_handler.yield_split(oversample=False)):
    train_meta = dset.train.metadata
    valid_meta = dset.validation.metadata

    # Whatever you want to inspect
    train_epirrs = {train_meta[md5][EPIRR] for md5 in dset.train.ids}
    valid_epirrs = {valid_meta[md5][EPIRR] for md5 in dset.validation.ids}
    train_uuids = {train_meta[md5][UUID] for md5 in dset.train.ids}
    valid_uuids = {valid_meta[md5][UUID] for md5 in dset.validation.ids}

    train_classes = Counter(dset.train.original_labels)
    valid_classes = Counter(dset.validation.original_labels)

    print(f"--- Fold {i} ---")
    print(
        f"Train: {len(dset.train.ids)} samples, {len(train_uuids)} UUIDs, {len(train_epirrs)} EpiRRs"
    )
    print(
        f"Valid: {len(dset.validation.ids)} samples, {len(valid_uuids)} UUIDs, {len(valid_epirrs)} EpiRRs"
    )

    print()
    for label in classes:
        # print(
        #     f"Class '{label}': Train {train_classes[label]} samples, Valid {valid_classes[label]} samples"
        # )
        empty_train = train_classes[label] == 0
        empty_valid = valid_classes[label] == 0
        if empty_train:
            print(f"WARNING: Class '{label}' has no training samples!")
        if empty_valid:
            print(f"WARNING: Class '{label}' has no validation samples!")
        if any([empty_train, empty_valid]):
            print(
                f"Class '{label}': Train {train_classes[label]} samples, Valid {valid_classes[label]} samples"
            )
    print()
    if len(train_epirrs & valid_epirrs) > 0:
        print("WARNING: EpiRR leakage detected!")
        print(f"EpiRR leakage: {train_epirrs & valid_epirrs}")
    if len(train_uuids & valid_uuids) > 0:
        print("WARNING: UUID leakage detected!")
        print(f"UUID leakage: {train_uuids & valid_uuids}")
    print()

In [ ]:
dset = dataset_handler.train_val_dset
uuids = np.array([dset.metadata[md5][UUID] for md5 in dset.ids])
uuids_unique, uuids_inverse = np.unique(uuids, return_inverse=True)
labels_unique = [dset.encoded_labels[uuids == uuid][0] for uuid in uuids_unique]

# Reproduce the EpiRR grouping
uuid_epirr = {}
for md5 in dset.ids:
    meta = dset.metadata[md5]
    uuid_epirr[meta[UUID]] = meta[EPIRR]
epirr_per_uuid = [uuid_epirr[u] for u in uuids_unique]
_, epirr_groups = np.unique(epirr_per_uuid, return_inverse=True)

# Decode labels for display
classes_list = dataset_handler.classes
label_decoder = dict(enumerate(classes_list))

skf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
for i, (train_idx, valid_idx) in enumerate(
    skf.split(
        X=np.empty((len(uuids_unique), 1)),
        y=labels_unique,
        groups=epirr_groups,
    )
):
    train_labels = np.array(labels_unique)[train_idx]
    valid_labels = np.array(labels_unique)[valid_idx]

    # Pre-oversampling UUID counts per class
    pre_counts = Counter(label_decoder[l] for l in train_labels)

    # Oversampling in UUID space
    ros = RandomOverSampler(random_state=42)
    resampled_uuids, resampled_labels = ros.fit_resample(
        np.array(uuids_unique[train_idx]).reshape(-1, 1),
        train_labels,
    )
    post_counts = Counter(label_decoder[l] for l in resampled_labels)

    print(f"--- Fold {i} ---")
    print(f"{'Class':<45} {'Pre':>5} {'Post':>5} {'Valid':>5}")
    for label in sorted(classes_list):
        enc = {v: k for k, v in label_decoder.items()}[label]
        v_count = np.sum(valid_labels == enc)
        print(f"{label:<45} {pre_counts[label]:>5} {post_counts[label]:>5} {v_count:>5}")
    print()